# TinyDB Chat Room

### Concept
A database for a chat room that tracks **users and messages** and updates as users create posts.

### Databases

1. **Users**
   - User handle
   - User age
   - Post counter

2. **Posts**
   - User handle
   - User age
   - Post
   - Time of post

- I chose to use two databases to keep the users database lighter.
- I added user handle and user age to Posts databse refrencable to the users DB. 

### Functions

- **`create_user()`**
  - Inserts a new user into `Users.json`.

- **`match_user()`**
  - Uses a compound query to check whether a user exists.
  - Checks both `user_handle` and `user_age` to verify the user.
  - I used `where()` instead of `Query()` and `matches()` so I could make the handle check case-insensitive.

- **`create_post()`**
  - Verifies the user.
  - Inserts the post into `Posts.json`.
  - Increments that user's `post_counter` by `1` in `Users.json`.

# Testing: I added 100 users and 10k posts to see how TinyBD will handle it. 
- It took 51 seconds to add users, posts and update the post count. - Seems reasonable. 
- It took less than 1 second for simple query I made for this DB. - Passed.



# Create DB and users: 

In [52]:
from tinydb import TinyDB, Query

In [53]:
posts = TinyDB('Posts.json')
users = TinyDB('Users.json')

In [68]:
# Insert new user into Users.json
from tinydb import where # this is instead of inti queary tool
import re

def create_user():
    user_handle = input("Enter user handle: ")

    # check if user handle exsits for database cleanliness 
    matched_user = users.search(   
        where('user_handle').matches(
            user_handle,
            flags=re.IGNORECASE
        )
    )

    # If the search found a user, the handle is already taken
    if matched_user:
        print("User handle taken.")
        return

    user_age = int(input("Enter user age: "))

    interests = input(
        "Enter two interests separated by a comma with no space "
        "(e.g. sports,python): "
    ).split(",")

    users.insert({
        "user_handle": user_handle,
        "user_age": user_age,
        "interests": interests,
        "post_count": 0
    })

    print(f"Welcome {user_handle} !!!")
    return None

In [70]:
# testing create_user 
create_user()

Enter user handle:  sam


User handle taken.


In [65]:
# Checing users.json:
users.all()

[{'user_handle': 'yis',
  'user_age': 30,
  'interests': ['Tennis', 'Coding'],
  'post_count': 0},
 {'user_handle': 'Dan',
  'user_age': 35,
  'interests': ['Tennis', 'Python'],
  'post_count': 0},
 {'user_handle': 'Sam',
  'user_age': 12,
  'interests': ['Candy', 'Sports'],
  'post_count': 0}]

Added users sucessfully 

# Multi-item qeury Users.json for matching users:

In [72]:
# making a multi step query for my post creation stage: 

def match_users(user_handle,user_age): # checks if user_handle exists while having a matching age. 
    matched_user = users.search(
        (where("user_handle").matches(user_handle,flags=re.IGNORECASE)) &  # using matches() so I can use ignorecase
        (where("user_age") == user_age)
    )
    if not matched_user:
        print("User or age incorrect")
        return  

    return matched_user
    
match_users("sam",12)

[{'user_handle': 'Sam',
  'user_age': 12,
  'interests': ['Candy', 'Sports'],
  'post_count': 0}]

Create users function is working. 

This needed to query my DB for two features to insure a correct match 

# Create Posts, and update both databases: 

In [67]:
# Create post function: 

from datetime import datetime
from tinydb.operations import increment

def create_post():
    
    user_handle = input("Enter user handle: ")
    user_age = int(input("Enter user age: "))

    matched_users = match_users(user_handle, user_age)
    
    if not matched_users: 
        return
    # if correct info was entered get post and update Posts.json
    post = input("Enter your post: ")
    time = datetime.now().isoformat() # iso format so it can be inserted into a json
    posts.insert({
        "user_handle":user_handle,
        "user_age":user_age,
        "post":post,
        "time":time})
    # +1 to post counter: 
    users.update(
        increment('post_count'),
        (where('user_handle').matches(user_handle,flags=re.IGNORECASE)) &  # using matches() so I can use ignorecase
        (where('user_age') == user_age))
    
    print(f"\n Posted your message:\n\n{post}")    
    return None


create_post()

Enter user handle:  yis
Enter user age:  30
Enter your post:  Hi everyone! 



 Posted your message:

Hi everyone! 


### create_post function review: 

This step requires multiple complex queries and to update both data bases

1. query that user exsists with matching age.
2. insert post and user info into Posts.json, that way posts can be tracked back to users later.  
3. find matching user in Users DB again, and update post count using tinyDB's increment tool



# Check DB to see if its working: 

In [74]:
posts.all()

[{'user_handle': 'yis',
  'user_age': 30,
  'post': 'Hi everyone! ',
  'time': '2026-09-30T17:55:27.086029'}]

We can see that posts are getting added each time in order

In [75]:
users.all()

[{'user_handle': 'yis',
  'user_age': 30,
  'interests': ['Tennis', 'Coding'],
  'post_count': 1},
 {'user_handle': 'Dan',
  'user_age': 35,
  'interests': ['Tennis', 'Python'],
  'post_count': 0},
 {'user_handle': 'Sam',
  'user_age': 12,
  'interests': ['Candy', 'Sports'],
  'post_count': 0}]

We can see that users post count is getting added up properly

# Populating my Database to see limitation - scalability 

In [76]:
import random
import time
from datetime import datetime


def add_test_data():

    start_time = time.perf_counter()

    interests_list = [
        "sports",
        "python",
        "music",
        "movies",
        "gaming",
        "fitness",
        "travel",
        "reading"
    ]

    # Add 100 users
    for i in range(100):
        users.insert({
            "user_handle": f"user{i}",
            "user_age": random.randint(18, 70),
            "post_count": 0,
            "interests": random.sample(interests_list, 2)
        })

    # Get all users
    all_users = users.all()

    # Add 10,000 posts
    for i in range(10000):

        user = random.choice(all_users)

        posts.insert({
            "user_handle": user["user_handle"],
            "user_age": user["user_age"],
            "post": f"Test post {i}",
            "time": datetime.now().isoformat()
        })

        users.update(
            increment("post_count"),
            where("user_handle") == user["user_handle"]
        )

    end_time = time.perf_counter()

    elapsed_time = end_time - start_time

    print("Added 100 users and 10,000 posts.")
    print(f"Time taken: {elapsed_time:.2f} seconds")


add_test_data()  # only run once

Added 100 users and 10,000 posts.
Time taken: 51.57 seconds


adding 100 users and 10k posts and 10k post counter updates took almost a minute, 

which isn't terrible.

# Testing the DB: 

In [77]:
# Timing getting all posts
# Timing getting all users
import time
from tinydb import where

def get_all_users():
    start_time = time.perf_counter()
    
    users.all()

    end_time = time.perf_counter()

    elapsed_time = end_time - start_time
    return elapsed_time

    
def get_all_posts():
    start_time = time.perf_counter()
    
    posts.all()

    end_time = time.perf_counter()

    elapsed_time = end_time - start_time
    return elapsed_time

print(f'time to get all users {get_all_users()}')
print(f'time to get all posts {get_all_posts()}')

time to get all users 0.0006146670020825695
time to get all posts 0.023359708000498358


Timing seems reasonable

In [78]:


# Time it takes to find all posts from one user

start = time.perf_counter()

user_posts = posts.search(
    where("user_handle") == "user5"
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(user_posts)}")

Time: 0.021265 seconds
Posts found: 87


In [79]:
# Time it takes to find all posts from users with age 30

start = time.perf_counter()

age_30_posts = posts.search(
    where("user_age") == 30
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(age_30_posts)}")

Time: 0.021451 seconds
Posts found: 197


In [81]:
# Time it takes to find all posts from users between ages 20-40

start = time.perf_counter()

age_20_40_posts = posts.search(
    (where("user_age") >= 20) &
    (where("user_age") <= 40)
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(age_20_40_posts)}")

Time: 0.024362 seconds
Posts found: 3588


### All tests had sufficant speeds for my use case. The DB performed in reasonable time. 

Lets see how it does with even more complex searching and functinos. 

In [83]:
posts.all()

[{'user_handle': 'yis',
  'user_age': 30,
  'post': 'Hi everyone! ',
  'time': '2026-09-30T17:55:27.086029'},
 {'user_handle': 'user72',
  'user_age': 57,
  'post': 'Test post 0',
  'time': '2026-09-30T18:05:30.337302'},
 {'user_handle': 'user35',
  'user_age': 50,
  'post': 'Test post 1',
  'time': '2026-09-30T18:05:30.337765'},
 {'user_handle': 'user1',
  'user_age': 19,
  'post': 'Test post 2',
  'time': '2026-09-30T18:05:30.338055'},
 {'user_handle': 'user37',
  'user_age': 20,
  'post': 'Test post 3',
  'time': '2026-09-30T18:05:30.338319'},
 {'user_handle': 'user81',
  'user_age': 67,
  'post': 'Test post 4',
  'time': '2026-09-30T18:05:30.338580'},
 {'user_handle': 'user13',
  'user_age': 69,
  'post': 'Test post 5',
  'time': '2026-09-30T18:05:30.338874'},
 {'user_handle': 'user20',
  'user_age': 64,
  'post': 'Test post 6',
  'time': '2026-09-30T18:05:30.339150'},
 {'user_handle': 'user79',
  'user_age': 61,
  'post': 'Test post 7',
  'time': '2026-09-30T18:05:30.339422'},
 {'